| Library       | Primary Use Case        | Key Features                                         | Strengths                                                                 | Limitations                                      | Best For                                |
|---------------|-------------------------|------------------------------------------------------|---------------------------------------------------------------------------|--------------------------------------------------|-----------------------------------------|
| **pypdf**     | General PDF manipulation| Merge, split, rotate, metadata, text extraction      | Lightweight, pure Python, good for metadata and PDF operations            | Basic text extraction, struggles with complex layouts | Simple tasks, metadata handling          |
| **pdfplumber**| Text + layout extraction| Extract text with coordinates, tables, images        | Accurate text extraction, supports tables and positional data             | Heavier dependency, slower on large PDFs         | Structured text, tables, precise layout  |
| **docling**   | Rich document parsing   | Semantic paragraphs, tables, metadata, modern API    | Advanced parsing, structured document model, ideal for AI/RAG pipelines   | Newer library, smaller ecosystem, less documentation | AI pipelines, semantic parsing, RAG use  |


In [1]:
# Install the required libraries
%pip install pypdf pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pypdf
import pdfplumber
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


Focus: Reading metadata, extracting raw text, and counting pages.

Extracting Metadata and Page Count

In [3]:
# Path to your PDF file (replace with your file)
pdf_path = "fepr101.pdf"

with open(pdf_path, "rb") as f:
    reader = pypdf.PdfReader(f)
    
    # Get basic info
    num_pages = len(reader.pages)
    metadata = reader.metadata
    
    print(f"Total Pages: {num_pages}")
    print("--- Metadata ---")
    for key, value in metadata.items():
        print(f"{key}: {value}")

Total Pages: 38
--- Metadata ---
/CreationDate: D:20241209154755+05'30'
/Creator: Adobe InDesign CS6 (Windows)
/ModDate: D:20260327113130+05'30'
/Producer: Adobe PDF Library 10.0.1
/Trapped: /False


In [22]:
# Extract text from the first page
engBook = "fepr101.pdf"
with open(engBook, "rb") as f:
    reader = pypdf.PdfReader(f)
    first_page = reader.pages[1:4]
    texts = [page.extract_text() for page in first_page]
    # text = first_page.extract_text()
    print("--- First Page Text ---")
    print(texts) # Showing first 500 characters
    for i, page in enumerate(texts):
        print(f"--- Page {i+1} ---")
        print(page[:500])  # Show first 500 characters of each page

--- First Page Text ---
['2\nPoorvi—Grade 6\n Let us read\nI\nRama Natha was the son of a rich landlord. \nHis father left him large tracts of land when he \ndied. But Rama Natha did not spend even one \nday looking after his land. This was because \nhe had a funny idea—he believed there was \na magic potion that could turn any object \ninto gold. He spent all his time to learn more \nabout this potion. People cheated him often, \npromising to tell him about it, but he did not \ngive up. His wife, Madhumati, was tired of this \nand also worried because she saw how much \nmoney Rama Natha was spending. She was \nsure that soon they would be without money.\nOne day, a famous sage called Mahipati \ncame to their town. Rama Natha became his \nfollower and asked him about the potion. To his \nsurprise the sage answered, “Yes, in my travels \nin the Himalayas, I heard how you could make \nsuch a potion. But it is difficult.”\n“Tell me!” requested Rama Natha, not \nbelieving his luck.\n“You h

In [24]:
with open("engBook.txt", "w") as f:
    # Save the extracted text to a new file
    f.write("\n".join(texts))
 # Show first 500 characters of each page

Layout-Preserving Text Extraction


In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [26]:


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=15
)

chunks = text_splitter.split_text("\n".join(texts))

len(chunks)  # Check how many chunks were created


21

In [27]:
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
embedding_function = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5265.07it/s]


In [9]:
from langchain_core.documents import Document

In [29]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="engBook_collection",
    embedding_function=embedding_function,
    persist_directory="./engBook_db",
)

In [11]:
metadata = [{"source": f"page_{i+1}", "book": "english"} for i in range(len(chunks))]

vector_store.add_documents(
    documents=[Document(page_content=text, metadata=metadata) for text, metadata in zip(chunks, metadata)])

['e0a53ba8-431b-4328-8fe0-befff9d1fd66',
 '11e905c6-0d8a-49ec-b166-014e3d0c1bc8',
 'e721fd6c-721e-4587-9a79-6b85ba749625',
 '8bb99b55-3ef0-4884-a036-8277276e495b',
 'a708bed1-7a19-42cc-8da0-3ec128e7925a',
 '082bff2b-0c4a-4efa-993b-017db2692b3a',
 'e3992e43-b0b1-457e-827f-dcd74bbe9bd0',
 'da5357f5-86d4-4ba6-8732-ba52f982e1d1',
 '70e133b4-036d-4f65-bace-b79400a3d2d4',
 'a3c0ffc9-34bd-431c-b0e2-c05bacc55b26',
 '8004af46-b97a-4266-b1fe-ef81907daecd',
 'e2bca185-73b5-4808-9b6a-855541ece341',
 '6091a259-2a00-4433-98ea-e5d68066d08f',
 '415b741b-8499-40b5-b80f-bd7a0b8c7169',
 'a3a89eab-d723-46a2-a7fa-901ba735ab68',
 '6bb09a26-5fe7-419b-81ba-a539a54bb4b5',
 '90d267c3-df3c-48ee-81c3-82dbb511546b',
 'fa412e67-2f1c-47eb-b961-0e2e410683df',
 '19375575-6811-43c7-a9d8-77cf5e1efa61',
 'b6505b8f-3221-4a05-928b-2c8cc47ee96d',
 'f8066226-b5fe-4b43-91b2-875998bcdaae']

In [12]:
results = vector_store.similarity_search(
    "who was Rama Natha?",
    k=4,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Let us discuss
 1. What did Rama Natha believe?
 2. How did the sage help Rama Natha?
 3. Do you think [{'book': 'english', 'source': 'page_15', 'attempt': 2}]
* Let us discuss
 1. What did Rama Natha believe?
 2. How did the sage help Rama Natha?
 3. Do you think [{'attempt': 2, 'book': 'english', 'source': 'page_15'}]
* Let us discuss
 1. What did Rama Natha believe?
 2. How did the sage help Rama Natha?
 3. Do you think [{'source': 'page_15', 'book': 'english', 'attempt': '2'}]
* Let us discuss
 1. What did Rama Natha believe?
 2. How did the sage help Rama Natha?
 3. Do you think [{'attempt': 2, 'book': 'english', 'source': 'page_15'}]


In [13]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

# Load API keys from .env file
load_dotenv()

# Try to configure Gemini
gemini_api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if gemini_api_key:
    genai.configure(api_key=gemini_api_key)
    print("Gemini API configured successfully.")
else:
    print("Warning: GOOGLE_API_KEY or GEMINI_API_KEY not found in environment variables.")

Gemini API configured successfully.


C:\Users\Ayush Mohanty\AppData\Local\Temp\ipykernel_24952\2939143812.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [14]:
# Use a stable Gemini model
model_name = "gemini-3.5-flash"
if gemini_api_key:
    gemini_Client = genai.GenerativeModel(model_name)
    print(f"Gemini client initialized with model: {model_name}")
else:
    gemini_Client = None
    print("Gemini client not initialized (missing API key).")

Gemini client initialized with model: gemini-3.5-flash


In [15]:
question = "Who was Rama Natha? tell me about him."
results = vector_store.similarity_search(
    question,
    k=5,
)
context = " ".join([res.page_content for res in results])


In [16]:
if gemini_Client:
    response = gemini_Client.generate_content(f"""You are an AI tutor that answers questions from English book lessons.
Question: 
{question}

Context:                                       
{context}
                                          
Instructions:
- Use ONLY the information from the provided lesson context to answer.
- If the answer is explicitly stated in the lesson, quote or paraphrase it clearly.
- If the answer requires interpretation, explain it in simple, student-friendly language.
- Do NOT invent information outside the lesson context.
- If the context does not contain the answer, say: \"The lesson does not provide enough information to answer this question.\"
- Keep the answer concise, accurate, and easy to understand.
""")
else:
    # Fallback to Groq since the user has GROQ_API_KEY
    groq_api_key = os.environ.get("GROQ_API_KEY")
    if groq_api_key:
        print("Using Groq fallback as Gemini API key is missing...")
        from langchain_groq import ChatGroq
        from langchain_core.prompts import ChatPromptTemplate
        
        llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)
        system_prompt = (
            "You are an assistant for question-answering tasks. "
            "Use the following pieces of retrieved context to answer the question. "
            "If you don't know the answer, say that you don't know. "
            "Use three sentences maximum and keep the answer concise."
            "\n\n"
            "{context}"
        )
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{input}"),
        ])
        
        prompt = prompt_template.format_messages(context=context, input=question)
        response = llm.invoke(prompt)
    else:
        raise ValueError("Neither Gemini nor Groq API keys found. Please set GOOGLE_API_KEY or GROQ_API_KEY in your .env file.")

In [17]:
if gemini_Client:
    print(response.text)
else:
    # Groq response
    print(response.content)

The lesson does not provide enough information to answer this question.


In [18]:

# pdfplumber is much better at keeping words in their visual order
with pdfplumber.open(pdf_path) as pdf:
    # Get the first page
    first_page = pdf.pages[0]
    
    # Extract text with a layout approximation
    structured_text = first_page.extract_text(layout=True)
    print("--- Structured Text Layout ---")
    print(structured_text)

--- Structured Text Layout ---
                                                                         
                                                                         
                                                                         
                                                                         
                                                       U        1        
                                                          nit            
                                                                         
                                                                         
                   FabLes     and    FoLk    TaLes                       
                                                                         
                                                                         
                                                                         
                                                                         
       

Table Extraction to Pandas DataFrame

In [19]:
# Extracting tabular data effortlessly
with pdfplumber.open(pdf_path) as pdf:
    # Loop through pages to find a table (checking page 1 here as an example)
    page = pdf.pages[0] 
    tables = page.extract_tables()
    
    if tables:
        print(f"Found {len(tables)} table(s) on this page.")
        # Convert the first table found into a clean Pandas DataFrame
        df = pd.DataFrame(tables[0][1:], columns=tables[0][0])
        display(df.head())
    else:
        print("No tables detected on this page. Try a page that contains a grid or table.")

No tables detected on this page. Try a page that contains a grid or table.


Focus: Cropping specific regions (Bounding Boxes), filtering text, and page manipulation.

Visual Cropping / Bounding Box Extraction

In [20]:
# Extracting text from a specific geometric area of the page (e.g., a header or sidebar)
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    
    # Define a bounding box (x0, top, x1, bottom) 
    # Usually measured in points (1 inch = 72 points) from top-left corner
    # Let's crop the top 2 inches of the page
    bounding_box = (0, 0, page.width, 144) 
    
    cropped_page = page.within_bbox(bounding_box)
    cropped_text = cropped_page.extract_text()
    
    print("--- Extracted Text from Header Crop ---")
    print(cropped_text)

--- Extracted Text from Header Crop ---
U 1
nit
FabLes and FoLk TaLes


Page Manipulation & Filtering (Saving a new PDF)

In [21]:
# Combining pypdf and logic to filter and split files
writer = pypdf.PdfWriter()

with open(pdf_path, "rb") as f:
    reader = pypdf.PdfReader(f)
    
    # Advanced filter: Find and save only pages that mention a specific keyword
    keyword = "Invoice"
    
    for page_num in range(len(reader.pages)):
        page = reader.pages[page_num]
        text = page.extract_text()
        
        if keyword.lower() in text.lower():
            print(f"Found keyword on page {page_num + 1}. Adding to output...")
            writer.add_page(page)

# Save the filtered pages to a new file
output_filename = "filtered_pages.pdf"
with open(output_filename, "wb") as output_pdf:
    writer.write(output_pdf)

print(f"\nCreated '{output_filename}' successfully!")


Created 'filtered_pages.pdf' successfully!
